# La Liga season finish probabilities

This compact walkthrough uses the shared Dixon–Coles, Elo, optional odds, and scoreline simulation pipeline. It replaces the original simple Poisson notebook.

Install `pip install -e '.[notebook]'`, create `.env` from `.env.example`, and set `FOOTBALL_DATA_API_KEY`. `ODDS_API_KEY` is optional. Open this notebook from the repository root and run the cells in order. See [the README](README.md) for setup.


In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display

from standing_prediction.predict_positions import run_prediction

load_dotenv()


## Configure the forecast

`season=None` infers the current European season with a July boundary; use `2026` for 2026/27. Calibration is optional and trains on earlier seasons. Odds are used only when a dated observation is available before the forecast and kickoff.


In [ ]:
competition = "PD"
season = None
n_sim = 10000
seed = 7
use_odds = True
calibrate = False
out_dir = Path("out")


## Run the shared pipeline

The pipeline validates the full schedule, loads up to two prior seasons, fits time-decayed Dixon–Coles and Elo models, and simulates the remaining scorelines. Teams without observed history start at neutral strength. Each run saves the report, CSVs, provenance, and input snapshots.


In [ ]:
forecast = run_prediction(
    competition=competition,
    season=season,
    n_sim=n_sim,
    seed=seed,
    use_odds=use_odds,
    calibrate=calibrate,
    out_dir=out_dir,
)

print(f"Season start: {forecast.metadata['season']}")
print(f"Generated: {forecast.metadata['generated_at']}")
print(f"Data cutoff: {forecast.metadata['data_cutoff']}")
print(f"Odds coverage: {forecast.metadata['odds_matched']}/{forecast.metadata['remaining_fixtures']} remaining fixtures")


## Summary

Expected points are the simulated mean. The 10th–90th point percentiles reflect match uncertainty conditional on fitted team strengths, excluding uncertainty in those strengths and future squad changes. Event percentages describe the title, top four, and bottom three positions. These positions do not determine qualification or direct relegation in every league. `*_mc_se_pp` measures simulation sampling error in percentage points.


In [ ]:
display(forecast.summary.style.format("{:.2f}"))


## Final-position probabilities

Each row sums to 100%. The ranking uses the modeled league sporting criteria; fair-play and deciding playoff matches are not simulated. See [pipeline details](PIPELINE_DETAILS.md) for the exact supported criteria and historical evaluation.


In [ ]:
display(
    forecast.probabilities.style
    .format("{:.2f}%")
    .background_gradient(cmap="Greens", vmin=0, vmax=100)
    .set_properties(**{"text-align": "center"})
)

prefix = competition.lower()
for suffix in ("position_probs.html", "position_probs.csv", "summary.csv", "metadata.json"):
    print(out_dir / f"{prefix}_{suffix}")
print(f"Timestamped archives: {out_dir / 'snapshots' / competition.lower() / str(forecast.metadata['season'])}")
